In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset

# Verificamos GPU
# Asignamos automaticamente la tarjeta grafica si esta dispobible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Cargamos el dataset
print("Cargando dataset...")
hf_dataset = load_dataset("cnxiaomai/pokemon-classification-gen1-9")

POKEMON_TYPES = [
    'normal', 'fire', 'water', 'grass', 'electric', 'ice',
    'fighting', 'poison', 'ground', 'flying', 'psychic', 'bug',
    'rock', 'ghost', 'dragon', 'steel', 'fairy', 'dark'
]
type_to_idx = {t: i for i, t in enumerate(POKEMON_TYPES)}
num_types = len(POKEMON_TYPES)

# En train alteramos las imagenes. (Con Data Augmentation para reducir Overfitting)
train_transforms = transforms.Compose([
    transforms.Resize((299, 299)), # Dimensiones estandar
    transforms.RandomHorizontalFlip(p=0.5), # Espejamos las imagenes con 50% de probabilidad
    transforms.RandomRotation(degrees=15), #Giramos la imagen levemente
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Alteramos la iluminacion
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# En validacion no queremos alterar las imagenes, solo necesitamos la dimension y el formato correcto
val_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Clase dataset
# Adaptamos los datos del Hugging Face a PyTorch
class PokemonDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.hf_data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.hf_data) # Numero total de muestras en el dataset

# A significa Alpha (Canal Alfa). Este es un cuarto canal de información que se añade a la imagen,
# y su única función es determinar qué tan transparente u opaco es cada píxel.
# Convertimos RGBA a RGB ya que los modelos pre-entrenados esperan 3 canales

    def __getitem__(self, idx): # Dado un indice, busca los datos en esa posicion y los transforma
        item = self.hf_data[idx]
        image = item['image_data'].convert('RGBA').convert('RGB')

        if self.transform:
            image = self.transform(image)

        type_str = str(item['Type 1']).lower().strip()
        label = type_to_idx.get(type_str, 0)

        return {'image': image, 'label': torch.tensor(label, dtype=torch.long)} # Devuelve imagen procesada y con etiqueta num en diccionario limpio

train_dataset = PokemonDataset(hf_dataset['train'], transform=train_transforms)
val_dataset = PokemonDataset(hf_dataset['test'], transform=val_transforms)

num_workers = min(4, os.cpu_count() or 2)

# Agrupamos los datos en conjuntos de 64 imagenes.
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True, # Mezclamos para que el modelo no memorice el orden
    num_workers=num_workers,
    pin_memory=True) # Ponemos los datos en un area bloqueada, GPU toma info de la RAM directamente

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True)



Usando dispositivo: cuda
Cargando dataset...


In [ ]:
# 5. MODELO + FINE-TUNING
weights = models.Inception_V3_Weights.DEFAULT
model = models.inception_v3(weights=weights)

# Congelamos primero todo
for param in model.parameters():
    param.requires_grad = False

# Descongelamos las capas profundas (Mixed_7c) para adaptar características
for param in model.Mixed_7c.parameters():
    param.requires_grad = True

# Reemplazamos la capa final
model.fc = nn.Linear(model.fc.in_features, num_types)
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_types)
model = model.to(device)

# 6. OPTIMIZADOR CON LEARNING RATE DIFERENCIADO
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': model.Mixed_7c.parameters(), 'lr': 1e-4}, # Despacio para no romper lo preentrenado
    {'params': model.fc.parameters(), 'lr': 1e-3},       # Ritmo normal para el clasificador
    {'params': model.AuxLogits.fc.parameters(), 'lr': 1e-3}
])

# 7. ENTRENAMIENTO
# Scheduler: reduce el Learning Rate a la mitad cada 5 épocas
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

epochs = 15 # Aumentamos las épocas para darle tiempo a aprender

print("\n¡Empezando entrenamiento optimizado con Fine-Tuning!")

for epoch in range(epochs):
    print(f"\nÉpoca {epoch + 1}/{epochs}")
    print("-" * 20)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            loader = train_loader
            dataset_size = len(train_dataset)
        else:
            model.eval()
            loader = val_loader
            dataset_size = len(val_dataset)

        running_loss = 0.0
        running_corrects = 0

        for batch in loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                if phase == 'train':
                    outputs, aux_outputs = model(images)
                    loss_main = criterion(outputs, labels)
                    loss_aux = criterion(aux_outputs, labels)
                    loss = loss_main + 0.4 * loss_aux
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * images.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_size
        epoch_acc = running_corrects.double() / dataset_size

        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
    scheduler.step()


¡Empezando entrenamiento optimizado con Fine-Tuning!

Época 1/15
--------------------
Train Loss: 3.4197 Acc: 0.2514
Val Loss: 2.1750 Acc: 0.3368

Época 2/15
--------------------
Train Loss: 2.8722 Acc: 0.3808
Val Loss: 1.9728 Acc: 0.3901

Época 3/15
--------------------
Train Loss: 2.5245 Acc: 0.4740
Val Loss: 1.8720 Acc: 0.4265

Época 4/15
--------------------
Train Loss: 2.2560 Acc: 0.5538
Val Loss: 1.8097 Acc: 0.4478

Época 5/15
--------------------
Train Loss: 2.0062 Acc: 0.6280
Val Loss: 1.6223 Acc: 0.5174

Época 6/15
--------------------
Train Loss: 1.7270 Acc: 0.7113
Val Loss: 1.5371 Acc: 0.5383

Época 7/15
--------------------
Train Loss: 1.6011 Acc: 0.7504
Val Loss: 1.5871 Acc: 0.5383

Época 8/15
--------------------
Train Loss: 1.5393 Acc: 0.7646
Val Loss: 1.5481 Acc: 0.5494

Época 9/15
--------------------
Train Loss: 1.4705 Acc: 0.7865
Val Loss: 1.5508 Acc: 0.5522

Época 10/15
--------------------
Train Loss: 1.3839 Acc: 0.8114
Val Loss: 1.5302 Acc: 0.5597

Época 11/15
--